# 02 — TF-IDF + Logistic Regression Baseline (T131)

Classical-ML baseline for the Concierge intent classifier. Reads
`data/clinc150_mapped.csv` produced by T130 (01_label_taxonomy.ipynb).

**Splits**: trains on `split == "train"`, tunes `C` on `split == "val"`,
and evaluates on `split == "test"`. No leakage: the test split is touched
exactly once at the end; the val split is used only for hyperparameter
selection (not added to training data).

**Threshold**: `classifier_macro_f1 ≥ 0.80` (from `eval_thresholds.yaml`).

**Outputs**:
* `services/modelserver/artifacts/tfidf_logreg.joblib` — fitted pipeline
* `notebooks/results/tfidf_logreg_results.json` — metrics + artifact SHA-256

T134 (05_compare_and_export.ipynb) reads the results JSON and picks the
winning model across the three baselines (TF-IDF, small DL/ONNX, LLM zero-shot).

In [1]:
from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

LABELS: tuple[str, ...] = ("spam", "faq", "lead_intent", "escalate", "ambiguous")
DATA_CSV = Path("data/clinc150_mapped.csv")
ARTIFACT_PATH = Path("../services/modelserver/artifacts/tfidf_logreg.joblib")
RESULTS_PATH = Path("results/tfidf_logreg_results.json")

In [2]:
df = pd.read_csv(DATA_CSV)
train = df[df["split"] == "train"].reset_index(drop=True)
val = df[df["split"] == "val"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
print(f"train: {len(train):5d}   val: {len(val):5d}   test: {len(test):5d}")
assert set(train["label"].unique()) == set(LABELS), "training data missing a label"
assert set(val["label"].unique()) == set(LABELS), "val data missing a label"
assert set(test["label"].unique()) == set(LABELS), "test data missing a label"

train:  4550   val:   960   test:  2290


In [3]:
def build_pipeline(C: float) -> Pipeline:
    """Fresh TF-IDF + LogReg pipeline. Vectorizer params are fixed per spec;
    only the LogReg `C` is tuned.

    Note: `multi_class="auto"` from the original spec was removed in
    scikit-learn 1.7. Default multi-class behavior is now always auto.
    """
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=C,
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs",
        )),
    ])

In [4]:
# (1) + (2) Fit on train only, score on val, pick best C.
tuning: list[dict[str, float]] = []
for C in [0.1, 1.0, 10.0]:
    pipe = build_pipeline(C)
    pipe.fit(train["text"], train["label"])
    val_preds = pipe.predict(val["text"])
    val_macro_f1 = f1_score(val["label"], val_preds, labels=list(LABELS), average="macro")
    tuning.append({"C": C, "val_macro_f1": float(val_macro_f1)})
    print(f"C={C:>5}  val macro-F1: {val_macro_f1:.4f}")

best = max(tuning, key=lambda r: r["val_macro_f1"])
print(f"\nBest C: {best['C']}  (val macro-F1: {best['val_macro_f1']:.4f})")

C=  0.1  val macro-F1: 0.8390


C=  1.0  val macro-F1: 0.8897


C= 10.0  val macro-F1: 0.8989

Best C: 10.0  (val macro-F1: 0.8989)


In [5]:
# (3) Refit the chosen pipeline on the train split (no leakage from val),
# then evaluate exactly once on the held-out test split.
pipeline = build_pipeline(best["C"])
pipeline.fit(train["text"], train["label"])

test_preds = pipeline.predict(test["text"])
test_macro_f1 = f1_score(test["label"], test_preds, labels=list(LABELS), average="macro")
per_class_f1_arr = f1_score(test["label"], test_preds, labels=list(LABELS), average=None)
per_class_f1 = {label: float(score) for label, score in zip(LABELS, per_class_f1_arr)}

print(f"Test macro-F1: {test_macro_f1:.4f}\n")
print("Per-class F1:")
for label, score in per_class_f1.items():
    print(f"  {label:12s} {score:.4f}")

Test macro-F1: 0.7620

Per-class F1:
  spam         0.5519
  faq          0.7333
  lead_intent  0.7317
  escalate     0.9062
  ambiguous    0.8869


In [6]:
print(classification_report(test["label"], test_preds, labels=list(LABELS), digits=4))

              precision    recall  f1-score   support

        spam     0.9871    0.3830    0.5519      1000
         faq     0.5834    0.9867    0.7333       450
 lead_intent     0.5875    0.9694    0.7317       360
    escalate     0.8529    0.9667    0.9062       270
   ambiguous     0.8299    0.9524    0.8869       210

    accuracy                         0.7148      2290
   macro avg     0.7682    0.8516    0.7620      2290
weighted avg     0.8147    0.7148    0.6883      2290



In [7]:
cm = confusion_matrix(test["label"], test_preds, labels=list(LABELS))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{lbl}" for lbl in LABELS],
    columns=[f"pred_{lbl}" for lbl in LABELS],
)
print(cm_df)

                  pred_spam  pred_faq  pred_lead_intent  pred_escalate  \
true_spam               383       304               237             41   
true_faq                  1       444                 3              0   
true_lead_intent          4         4               349              1   
true_escalate             0         3                 4            261   
true_ambiguous            0         6                 1              3   

                  pred_ambiguous  
true_spam                     35  
true_faq                       2  
true_lead_intent               2  
true_escalate                  2  
true_ambiguous               200  


In [8]:
# (4) Single-prediction latency over 1000 samples (production handles one
# visitor message at a time, so we time predict([t]) in a loop rather than
# batch predict).
sample_texts = test["text"].sample(1000, replace=True, random_state=42).tolist()

start = time.perf_counter()
for t in sample_texts:
    pipeline.predict([t])
elapsed_s = time.perf_counter() - start

latency_ms = (elapsed_s / len(sample_texts)) * 1000
print(f"Total time for {len(sample_texts)} predictions: {elapsed_s:.3f}s")
print(f"Mean per prediction: {latency_ms:.3f} ms")

Total time for 1000 predictions: 2.566s
Mean per prediction: 2.566 ms


In [9]:
# (6) Persist fitted pipeline. joblib uses pickle under the hood; the
# modelserver (T148) re-loads and verifies the SHA-256 at boot.
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, ARTIFACT_PATH)
size_kb = ARTIFACT_PATH.stat().st_size / 1024

# (7) SHA-256 of the artifact.
artifact_sha256 = hashlib.sha256(ARTIFACT_PATH.read_bytes()).hexdigest()
print(f"Saved:   {ARTIFACT_PATH.resolve()}")
print(f"Size:    {size_kb:,.1f} KB")
print(f"SHA-256: {artifact_sha256}")

Saved:   C:\Users\ahmad\OneDrive\Desktop\AIE bootcamp\week8_project\CONCIERGE\services\modelserver\artifacts\tfidf_logreg.joblib
Size:    995.3 KB
SHA-256: 8eeee9d254cddcb6190031c28b109e5c035c4ac2acdcaa3725f48be98ed7c557


In [10]:
# (5) Cost: classical ML, no API calls.
COST_PER_1K = 0.00

# (8) Results dict consumed by T134 (compare/export).
results = {
    "model": "tfidf_logreg",
    "macro_f1": float(test_macro_f1),
    "per_class_f1": per_class_f1,
    "latency_ms_per_prediction": float(latency_ms),
    "cost_per_1k_predictions": COST_PER_1K,
    "artifact_path": str(ARTIFACT_PATH).replace("\\", "/"),
    "artifact_sha256": artifact_sha256,
}
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))

{
  "model": "tfidf_logreg",
  "macro_f1": 0.7619951251667114,
  "per_class_f1": {
    "spam": 0.5518731988472623,
    "faq": 0.7332782824112304,
    "lead_intent": 0.7316561844863732,
    "escalate": 0.90625,
    "ambiguous": 0.8869179600886918
  },
  "latency_ms_per_prediction": 2.5661474999942584,
  "cost_per_1k_predictions": 0.0,
  "artifact_path": "../services/modelserver/artifacts/tfidf_logreg.joblib",
  "artifact_sha256": "8eeee9d254cddcb6190031c28b109e5c035c4ac2acdcaa3725f48be98ed7c557"
}


## Threshold check & next

Compare `macro_f1` above against `classifier_macro_f1 ≥ 0.80` in
`eval_thresholds.yaml`. If under threshold, options before adding more data:
* widen `ngram_range` to (1, 3) or add character n-grams
* try `class_weight=None` (current balanced weighting penalizes the dominant `spam` class — may help or hurt)
* widen the C grid

Next: `03_small_dl_onnx.ipynb` (T132) trains a small DL model and exports to
ONNX. `04_llm_zero_shot.ipynb` (T133) runs an LLM zero-shot baseline.
`05_compare_and_export.ipynb` (T134) picks the winner across all three by
macro-F1, with latency/size/cost tiebreakers.